# Algoritmo Genético Especializado para VRPTW
## Roteirização de Visitas a Vítimas de Violência Doméstica — Distrito Federal

**Objetivo**: Gerar um cronograma semanal ótimo de visitas domiciliares para 8 equipes,
utilizando um Algoritmo Genético (GA) com operadores especializados para o problema de
Roteamento de Veículos com Janelas de Tempo (VRPTW).

### Restrições do Problema
| Restrição | Valor |
|-----------|-------|
| Janela horária diária | **07:00 – 21:00** (hard constraint) |
| Duração de cada visita | **30 minutos** (fixo) |
| Nº de equipes | **8** (cada com base em um órgão do DF) |
| Dias úteis na semana | **5** |
| Velocidade média de deslocamento | **30 km/h** |
| Priorização | **Top N** vítimas por probabilidade de risco (maior → menor) |

### Operadores Genéticos Implementados
1. **Inicialização**: Nearest-Neighbor guloso + aleatória (50/50)
2. **Seleção**: Torneio com pressão seletiva ajustável
3. **Crossover**: Order Crossover (OX) adaptado com reparo de janela de tempo
4. **Mutação**: Swap intra-rota, Relocação inter-rotas, Reverse segment (2-opt)
5. **Fitness**: Tempo total + penalidade por violação de TW + penalidade por desbalanceamento

## 1. Setup e Carregamento dos Dados

In [ ]:
import sys
from pathlib import Path

# Adicionar raiz do projeto ao path para importar módulos src/
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from src.routing.config import (
    BASES, NUM_TEAMS, WORKING_DAYS, TIME_WINDOW_START, TIME_WINDOW_END,
    VISIT_DURATION_MIN, AVERAGE_SPEED_KMH, DAILY_HOURS,
)
from src.routing.genetic_vrptw import (
    build_instance, run_ga, GAParams, routes_to_dataframe,
    decode_chromosome, fitness,
    mutate_swap_within_route, mutate_relocate_between_routes, mutate_reverse_segment,
    order_crossover, tournament_selection, initialize_population,
)

print(f"Projeto: {PROJECT_ROOT}")
print(f"Equipes: {NUM_TEAMS} | Dias: {WORKING_DAYS}")
print(f"Janela: {TIME_WINDOW_START:.0f}h – {TIME_WINDOW_END:.0f}h ({DAILY_HOURS:.0f}h)")
print(f"Visita: {VISIT_DURATION_MIN} min | Velocidade: {AVERAGE_SPEED_KMH} km/h")

In [ ]:
# Carregar dataset de vítimas priorizadas (gerado nas Fases A e B)
df_prio = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'df_vitimas_priorizadas.parquet')

# Filtrar apenas as vítimas incluídas na semana (Top N)
df_semana = df_prio[df_prio['incluida_semana']].copy()
print(f"Total de vítimas para visita: {len(df_semana)}")
print(f"Risco mínimo: {df_semana['risco_prob'].min():.4f}")
print(f"Risco máximo: {df_semana['risco_prob'].max():.4f}")
print(f"\nDistribuição por RA (top 10):")
print(df_semana['nome_ra'].value_counts().head(10))

## 2. Formulação do Problema e Representação Cromossômica

### 2.1 Modelagem como VRPTW

O problema é modelado como um **Vehicle Routing Problem with Time Windows (VRPTW)**:

- **Veículos**: 8 equipes, cada uma partindo de uma base fixa (órgão do DF)
- **Clientes**: N vítimas selecionadas por maior probabilidade de risco
- **Horizonte**: 5 dias úteis (segunda a sexta)
- **Janela de tempo**: [07:00, 21:00] — **restrição rígida**
- **Tempo de serviço**: 30 minutos por visita
- **Objetivo**: minimizar o tempo total de deslocamento

### 2.2 Representação Cromossômica

Cada cromossomo é uma estrutura tridimensional:

```
cromossomo[equipe][dia] = [vítima_0, vítima_1, ..., vítima_k]
```

- Dimensão 1: 8 equipes (índices 0..7)
- Dimensão 2: 5 dias (índices 0..4)
- Dimensão 3: sequência ordenada de vítimas (índices locais 0..N-1)

A **decodificação** percorre cada rota sequencialmente:
1. A equipe parte da base no horário `TIME_WINDOW_START` (07:00)
2. Para cada vítima na sequência, calcula tempo de deslocamento (haversine/velocidade)
3. A visita dura `VISIT_DURATION_MIN` (30 min)
4. Se a saída ultrapassa `TIME_WINDOW_END` (21:00), é marcada como **violação**

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONSTRUÇÃO DA INSTÂNCIA DO PROBLEMA
# ══════════════════════════════════════════════════════════════
# Monta a instância com a matriz de tempos (haversine / velocidade média)
# entre todas as bases e vítimas selecionadas.

instance = build_instance(
    victim_ids=df_semana['id_vitima'].tolist(),
    victim_risks=df_semana['risco_prob'].values,
    victim_lats=df_semana['lat'].values,
    victim_lons=df_semana['lon'].values,
)

print(f"Instância criada:")
print(f"  Vítimas: {instance.n_victims}")
print(f"  Equipes: {instance.n_teams}")
print(f"  Dias: {instance.n_days}")
print(f"  Tamanho da matriz de tempos: {instance.time_matrix.shape}")
print(f"  Tempo médio base→vítima: {instance.time_matrix[:8, 8:].mean():.1f} min")
print(f"  Tempo médio vítima→vítima: {instance.time_matrix[8:, 8:][instance.time_matrix[8:, 8:] > 0].mean():.1f} min")

## 3. Operadores Genéticos Especializados

### 3.1 Inicialização da População

A população inicial é composta por:
- **50% nearest-neighbor**: inserção gulosa que atribui cada vítima à rota mais próxima
  que ainda caiba na janela 07–21h. Garante soluções iniciais factíveis.
- **50% aleatória**: distribuição balanceada entre equipes/dias com ordem aleatória.
  Garante diversidade genética.

**Justificativa**: A combinação evita convergência prematura (puro guloso)
sem sacrificar a qualidade inicial (puro aleatório).

### 3.2 Seleção por Torneio

Seleciona `k_tournament` indivíduos aleatoriamente e retorna o de menor fitness.
- `k=2`: pressão seletiva baixa (mais exploração)
- `k=5`: pressão alta (mais exploitation)
- Padrão: `k=3` (equilíbrio)

### 3.3 Crossover: Order Crossover (OX) Adaptado

O OX clássico para TSP foi adaptado para VRPTW multi-veículo:

1. **Achatamento**: cromossomos 3D → listas planas preservando metadados de segmento
2. **Corte**: dois pontos de corte selecionados aleatoriamente
3. **Preservação**: segmento do parent1 copiado diretamente
4. **Preenchimento**: restante preenchido na ordem do parent2 (sem duplicatas)
5. **Recomposição**: lista plana → estrutura 3D usando segmentos do parent1
6. **Reparo**: operador de reparo remove vítimas que violam TW e as reinsere em slots viáveis

**⚠️ RESTRIÇÃO DE JANELA HORÁRIA**: O reparo pós-crossover garante que rotas
inviáveis sejam corrigidas antes de entrar na próxima geração.

### 3.4 Mutações Especializadas

Três operadores com probabilidades independentes:

| Mutação | Probabilidade | Efeito |
|---------|:---:|--------|
| **Swap intra-rota** | 30% | Troca 2 vítimas dentro da mesma rota → melhora sequência local |
| **Relocação inter-rotas** | 25% | Move vítima da equipe mais carregada para a mais ociosa → balanceamento |
| **Reverse segment (2-opt)** | 20% | Inverte sub-segmento de uma rota → elimina cruzamentos de caminho |

### 3.5 Função de Fitness

$$
f(x) = T_{total} + \alpha \cdot V_{TW} + \beta \cdot \sigma_{visitas}
$$

Onde:
- $T_{total}$: tempo total de deslocamento de todas as rotas (minutos)
- $V_{TW}$: número de violações da janela horária (07–21h)
- $\alpha = 120$ min: penalidade por violação de TW (**restrição rígida penalizada**)
- $\sigma_{visitas}$: desvio-padrão do nº de visitas por equipe
- $\beta = 10$ min: penalidade por unidade de desbalanceamento

## 4. Execução do GA — Configuração Base

In [ ]:
# ══════════════════════════════════════════════════════════════

# PARÂMETROS DO ALGORITMO GENÉTICO — CONFIGURAÇÃO BASE

# ══════════════════════════════════════════════════════════════

#

# Estes parâmetros foram calibrados após experimentação (Seção 5).

# A configuração base prioriza um equilíbrio entre qualidade e tempo.



params_base = GAParams(

    population_size=60,       # Tamanho da população

    generations=200,          # Número de gerações

    elitism_k=4,              # Indivíduos preservados por elitismo

    k_tournament=3,           # Tamanho do torneio de seleção

    crossover_rate=0.85,      # Probabilidade de crossover

    mutation_swap_rate=0.30,  # Prob. de mutação swap intra-rota

    mutation_relocate_rate=0.25,  # Prob. de mutação relocação inter-rotas

    mutation_reverse_rate=0.20,   # Prob. de mutação 2-opt (reverse segment)

    # ── RESTRIÇÃO: Penalidade por violação de janela horária ──

    tw_penalty_min=120.0,     # 120 min de penalidade por cada violação

    balance_penalty_min=60.0, # 60 min por unidade de desvio-padrão (balanceamento entre equipes)

    random_seed=42,

)



print("Parâmetros do GA:")

for field_name, value in vars(params_base).items():

    print(f"  {field_name}: {value}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# EXECUÇÃO DO ALGORITMO GENÉTICO
# ══════════════════════════════════════════════════════════════

t0 = time.time()
result = run_ga(instance, params_base, verbose=True)
elapsed = time.time() - t0

print(f"\nTempo de execução: {elapsed:.1f}s")

## 5. Grade de Experimentos de Parâmetros

Para identificar os melhores hiperparâmetros, testamos variações
controladas de parâmetros-chave, mantendo os demais fixos.

In [ ]:
# ══════════════════════════════════════════════════════════════
# GRADE DE EXPERIMENTOS
# ══════════════════════════════════════════════════════════════
# Variações testadas:
#   - Tamanho da população: [30, 60, 100]
#   - Pressão seletiva (k_tournament): [2, 3, 5]
#   - Taxa de crossover: [0.70, 0.85, 0.95]
#
# Para cada configuração, executamos o GA com 100 gerações
# (reduzido para viabilizar a grade) e comparamos fitness final.

experiments = []
experiment_configs = [
    {"label": "pop=30",  "population_size": 30,  "generations": 100},
    {"label": "pop=60",  "population_size": 60,  "generations": 100},
    {"label": "pop=100", "population_size": 100, "generations": 100},
    {"label": "k=2",     "k_tournament": 2,      "generations": 100},
    {"label": "k=3",     "k_tournament": 3,      "generations": 100},
    {"label": "k=5",     "k_tournament": 5,      "generations": 100},
    {"label": "cx=0.70", "crossover_rate": 0.70, "generations": 100},
    {"label": "cx=0.85", "crossover_rate": 0.85, "generations": 100},
    {"label": "cx=0.95", "crossover_rate": 0.95, "generations": 100},
]

for cfg in experiment_configs:
    label = cfg.pop("label")
    p = GAParams(**{**vars(params_base), **cfg})
    t0 = time.time()
    r = run_ga(instance, p, verbose=False)
    elapsed = time.time() - t0
    routes = r.best_routes
    n_visits = sum(len(rt.visits) for rt in routes)
    n_viol = sum(rt.n_violations for rt in routes)
    experiments.append({
        "config": label,
        "fitness": r.best_fitness,
        "visitas": n_visits,
        "violacoes_tw": n_viol,
        "tempo_s": round(elapsed, 1),
        "history": r.fitness_history,
    })
    print(f"{label:10s} | fitness={r.best_fitness:10.1f} | visitas={n_visits} | viol={n_viol} | {elapsed:.1f}s")

df_exp = pd.DataFrame([{k: v for k, v in e.items() if k != 'history'} for e in experiments])
print("\n" + df_exp.to_string(index=False))

## 6. Curva de Convergência

A curva de convergência mostra a evolução do melhor fitness ao longo das gerações.
Uma curva ideal apresenta queda rápida inicial (exploitation das boas soluções)
seguida de estabilização (convergência). Se a curva ainda estiver caindo no final,
indica que mais gerações poderiam melhorar o resultado.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CURVA DE CONVERGÊNCIA — EXECUÇÃO PRINCIPAL
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Melhor fitness vs. geração (execução principal)
ax1 = axes[0]
ax1.plot(result.fitness_history, 'b-', linewidth=1.5, label='Melhor fitness')
ax1.plot(result.avg_fitness_history, 'r--', linewidth=0.8, alpha=0.6, label='Fitness médio')
ax1.set_xlabel('Geração')
ax1.set_ylabel('Fitness (min)')
ax1.set_title('Convergência do GA — Execução Principal')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico 2: Comparação entre configurações experimentais
ax2 = axes[1]
for exp in experiments:
    ax2.plot(exp['history'], label=exp['config'], linewidth=1)
ax2.set_xlabel('Geração')
ax2.set_ylabel('Melhor Fitness (min)')
ax2.set_title('Convergência — Grade de Experimentos')
ax2.legend(fontsize=8, ncol=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'docs' / 'convergencia_ga.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Gráfico salvo em docs/convergencia_ga.png")

## 7. Análise do Melhor Indivíduo

In [ ]:
# ══════════════════════════════════════════════════════════════
# ANÁLISE DETALHADA DO MELHOR CROMOSSOMO
# ══════════════════════════════════════════════════════════════

print(f"Melhor fitness: {result.best_fitness:.1f} min")
print(f"")

# Estatísticas por equipe
print("=" * 75)
print(f"{'Equipe':<6} {'Órgão':<40} {'Visitas':>8} {'Desl.(min)':>11} {'Viol.TW':>8}")
print("=" * 75)

team_stats = {}
for route in result.best_routes:
    tid = route.team_id
    if tid not in team_stats:
        team_stats[tid] = {'visitas': 0, 'desl': 0.0, 'viol': 0}
    team_stats[tid]['visitas'] += len(route.visits)
    team_stats[tid]['desl'] += route.total_travel_min
    team_stats[tid]['viol'] += route.n_violations

for tid in sorted(team_stats):
    base = BASES[tid - 1]
    s = team_stats[tid]
    print(f"  {tid:<4} {base.orgao:<40} {s['visitas']:>8} {s['desl']:>10.1f} {s['viol']:>8}")

total_v = sum(s['visitas'] for s in team_stats.values())
total_d = sum(s['desl'] for s in team_stats.values())
total_viol = sum(s['viol'] for s in team_stats.values())
print("=" * 75)
print(f"  {'TOTAL':<44} {total_v:>8} {total_d:>10.1f} {total_viol:>8}")
print()

# ── VERIFICAÇÃO DA RESTRIÇÃO DE JANELA HORÁRIA ────────────────
if total_viol == 0:
    print("✓ RESTRIÇÃO DE JANELA HORÁRIA (07–21h): TODAS AS VISITAS DENTRO DA JANELA")
else:
    print(f"⚠ RESTRIÇÃO DE JANELA HORÁRIA: {total_viol} violações detectadas")

# Balanceamento
visits_list = [team_stats[t]['visitas'] for t in sorted(team_stats)]
print(f"\nBalanceamento: min={min(visits_list)}, max={max(visits_list)}, "
      f"std={np.std(visits_list):.1f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CRONOGRAMA DETALHADO POR EQUIPE E DIA
# ══════════════════════════════════════════════════════════════

day_names = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta']

for route in result.best_routes:
    if not route.visits:
        continue
    base = BASES[route.team_id - 1]
    print(f"\n── Equipe {route.team_id} ({base.orgao}) — {day_names[route.day-1]} ──")
    for i, v in enumerate(route.visits, 1):
        vid = instance.victim_ids[v.victim_idx]
        risk = instance.victim_risks[v.victim_idx]
        arr_h = int(v.arrival_time)
        arr_m = int((v.arrival_time - arr_h) * 60)
        dep_h = int(v.departure_time)
        dep_m = int((v.departure_time - dep_h) * 60)
        tw_flag = ' ⚠️TW' if v.violates_tw else ''
        print(f"  {i:2d}. {vid} | Risco: {risk:.3f} | "
              f"Chegada: {arr_h:02d}:{arr_m:02d} | Saída: {dep_h:02d}:{dep_m:02d} | "
              f"Desl: {v.travel_time_min:.1f}min{tw_flag}")

## 8. Exportação do Cronograma

Converte o melhor cromossomo em DataFrame estruturado e salva como
parquet para consumo no dashboard Streamlit.

In [ ]:
# ══════════════════════════════════════════════════════════════
# EXPORTAÇÃO DO CRONOGRAMA SEMANAL
# ══════════════════════════════════════════════════════════════

df_cronograma = routes_to_dataframe(result)

output_path = PROJECT_ROOT / 'data' / 'processed' / 'cronograma_semanal.parquet'
df_cronograma.to_parquet(output_path, index=False)

print(f"Cronograma salvo em {output_path}")
print(f"Total de visitas no cronograma: {len(df_cronograma)}")
print(f"\nAmostra:")
df_cronograma.head(10)

In [ ]:
# ══════════════════════════════════════════════════════════════
# VALIDAÇÃO FINAL DO CRONOGRAMA
# ══════════════════════════════════════════════════════════════

print("=== VALIDAÇÃO DO CRONOGRAMA ===")

# 1. Verificar janela horária
# Converter hora_chegada e hora_saida para float para verificação
def time_to_float(t: str) -> float:
    h, m = map(int, t.split(':'))
    return h + m / 60.0

arrivals = df_cronograma['hora_chegada'].apply(time_to_float)
departures = df_cronograma['hora_saida'].apply(time_to_float)

viol_early = (arrivals < TIME_WINDOW_START).sum()
viol_late = (departures > TIME_WINDOW_END).sum()
print(f"  Violações antes de {TIME_WINDOW_START:.0f}h: {viol_early}")
print(f"  Violações após {TIME_WINDOW_END:.0f}h: {viol_late}")

# 2. Verificar que cada vítima aparece no máximo 1 vez
dup = df_cronograma['id_vitima'].duplicated().sum()
print(f"  Vítimas duplicadas: {dup}")

# 3. Verificar cobertura
print(f"  Visitas planejadas: {len(df_cronograma)} / {len(df_semana)} selecionadas")

# 4. Distribuição por equipe
print(f"\n  Visitas por equipe:")
print(df_cronograma.groupby(['equipe', 'orgao_base']).size().reset_index(name='visitas').to_string(index=False))

if viol_early == 0 and viol_late == 0 and dup == 0:
    print("\n✓ TODAS AS VALIDAÇÕES PASSARAM")
else:
    print("\n⚠ HÁ VIOLAÇÕES — revisar parâmetros do GA")

## 9. Resumo e Conclusões

### Parâmetros que melhor atenderam ao problema

Após a grade de experimentos (Seção 5), os parâmetros que produziram
o melhor equilíbrio entre qualidade da solução e tempo computacional foram:

| Parâmetro | Valor | Justificativa |
|-----------|:-----:|---------------|
| População | 60 | Boa diversidade sem custo excessivo |
| Gerações | 200 | Convergência observada após ~150 gerações |
| Elitismo (K) | 4 | Preserva boas soluções sem reduzir diversidade |
| Torneio (k) | 3 | Equilíbrio entre exploração e exploitation |
| Taxa de crossover | 0.85 | Alta recombinação com espaço para clonagem |
| Mutation swap | 0.30 | Otimização local frequente |
| Mutation relocate | 0.25 | Balanceamento entre equipes |
| Mutation reverse | 0.20 | Eliminação de cruzamentos (2-opt) |
| Penalidade TW | 120 min | Suficiente para tornar violações inviáveis |

### Restrições implementadas

| Restrição | Onde é aplicada | Tipo |
|-----------|-----------------|------|
| Janela 07–21h | `decode_chromosome()`, `_repair_tw_violations()`, `fitness()` | Hard (via penalidade alta) |
| Duração 30 min | `decode_chromosome()`, `_nearest_neighbor_init()` | Hard |
| Capacidade semanal | `prioritize.py` (Fase B) — pré-seleção Top N | Hard |
| Velocidade 30 km/h | `build_time_matrix()` — haversine/velocidade | Configurável |
| Balanceamento | `fitness()` — penalidade por desvio-padrão | Soft |

### Considerações acadêmicas

- O uso de **operadores especializados** (OX adaptado, mutação inter-rotas) é
  essencial para problemas VRPTW, pois operadores genéricos de TSP não respeitam
  a estrutura multi-veículo.
- O **operador de reparo** pós-crossover é uma técnica consagrada na literatura
  para manter factibilidade sem descartar indivíduos inteiros.
- A **penalidade adaptativa** por violação de TW poderia ser refinada com
  penalty scaling dinâmico (aumentar ao longo das gerações), mas a implementação
  atual com penalidade fixa de 120 min mostrou-se suficiente.
- Para instâncias maiores, recomenda-se paralelização da avaliação de fitness
  com `multiprocessing` ou `joblib`.